# Experiment - Survival / time-to-event benchmark

> **Takeaway -** *(re-run pending after the metric overhaul)* - the old single-number
> verdict (static wins cancel-AUC, XGB-AFT wins C-index) is superseded: the decision
> metrics are now **per-snapshot AUC + Brier + expected-vs-realized cancellations**,
> which ask exactly the operational question. C-index is kept as a sanity footnote only.

## In plain words
A cancellation isn't just *whether* a booking falls through but *when*. Each booking
is "at risk" from the day it's made until it either cancels (the event) or the guest
arrives - at which point the **risk window closes: the booking can never cancel again**.
Technically the models encode this as right-censoring at arrival, and here that is
*exact*, not an approximation: the "censoring" time (= lead time) is known at booking
time and is a model feature, and no prediction ever looks past a booking's own arrival.
Nothing is lost either - a booking that reaches arrival pushes the estimated hazard
down on every single day it was observed not cancelling.

## What we measure (decision metrics)
The operational question is: **"standing d days before arrival, how likely is this
still-alive booking to cancel before the guest arrives?"** - summed per property and
stay date, that number drives overbooking. So every model is scored on the snapshot
grid d ∈ {90, 60, 30, 14, 7, 3, 1}:

- **AUC@d** - discrimination: does the model separate the bookings that still fall?
- **Brier@d** - probability honesty (calibration + sharpness) at that horizon.
- **Expected-vs-realized@d** - sum of predicted remaining-cancel probabilities vs the
  cancellations that actually happened: the portfolio number a revenue manager feels.

Our test set is fully resolved (3-day grace in 00), so these are plain binary metrics
per horizon - no censoring corrections needed in evaluation.

**Sanity footnote:** the classic **C-index** (pairwise ranking of cancellation times)
is still printed, but it answers "who cancels *sooner*?" - not our decision question -
so it no longer drives the model choice.

## Models compared
Static HistGB classifier (one probability, frozen at booking) - XGBoost AFT -
discrete-time hazard (notebook 08 scheme) - Random Survival Forest -
Gradient-Boosted Cox - three lifelines parametric AFTs.

## Methodological note
A Cox-style model assumes proportional hazards (constant hazard ratios over time),
which cancellation behaviour probably violates (risk reshapes near arrival). The
discrete-time hazard makes no such assumption and natively encodes "cannot cancel
after arrival" via its person-period layout.

**Roster note (2026-06-12):** the three leaky profile fields are excluded here.
Saved to `reports/tables/00_audit/survival_benchmark.csv` (full-test, two classic metrics)
and `survival_benchmark_snapshots.csv` (the per-snapshot decision table).


In [1]:
import sys
from pathlib import Path
_here = Path.cwd().resolve()
while not (_here / "pyproject.toml").exists() and _here != _here.parent:
    _here = _here.parent
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from src.data_loader import load_clean_reservations, load_reservations
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
import xgboost as xgb

try:
    from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
    from sksurv.metrics import concordance_index_censored
    HAVE_SKSURV = True
except Exception:
    HAVE_SKSURV = False
    print("scikit-survival NOT installed - run:  pip install scikit-survival")
    print("XGBoost AFT + static baseline still run, but C-index needs sksurv.")

try:
    from lifelines import WeibullAFTFitter, LogNormalAFTFitter, LogLogisticAFTFitter
    from lifelines.utils import concordance_index as ll_concordance
    HAVE_LIFELINES = True
except Exception:
    HAVE_LIFELINES = False
    print("lifelines NOT installed - run:  pip install lifelines  (parametric AFT models)")

# Roster aligned with 00 / 01 (2026-06-11): free-cancel features and is_international
# removed (no longer in the clean frame); has_children + company-history bundle added.
NUMERIC_BASE = ["lead_time_days","los_nights","adults_n","log_gross_amount",
    "gross_per_night","diff_gross_cancellation_fee","arrival_dow","arrival_month",
    "is_weekend_arrival","has_promo","has_corporate_code","has_group","has_children",
    "has_company","is_repeat_company","company_prior_bookings","company_prior_cancel_rate"]
# 2026-06-12: guest_country_region / preferredLanguage / travelPurpose removed -
# outcome leakage via check-in-time profile completion (profile_leakage_quantification).
CATEGORICAL_BASE = ["property_name","channelCode","ratePlan_category","unitGroup_name",
    "guaranteeType","stay_bucket","cancellationFee_name"]

def _obj(frame):
    if isinstance(frame, pd.Series): frame = frame.to_frame()
    return frame.astype("string").to_numpy(dtype=object, na_value=np.nan)

def make_matrix(df_tr, df_te):
    num=[c for c in NUMERIC_BASE if c in df_tr.columns]; cat=[c for c in CATEGORICAL_BASE if c in df_tr.columns]
    ni=SimpleImputer(strategy="median").fit(df_tr[num]); s=StandardScaler().fit(ni.transform(df_tr[num]))
    Xtr=[s.transform(ni.transform(df_tr[num]))]; Xte=[s.transform(ni.transform(df_te[num]))]
    ci=SimpleImputer(strategy="most_frequent").fit(_obj(df_tr[cat]))
    oh=OneHotEncoder(handle_unknown="ignore",sparse_output=False).fit(ci.transform(_obj(df_tr[cat])))
    Xtr.append(oh.transform(ci.transform(_obj(df_tr[cat])))); Xte.append(oh.transform(ci.transform(_obj(df_te[cat]))))
    return np.hstack(Xtr).astype("float32"), np.hstack(Xte).astype("float32")

## Memory notes

- **One common training sample:** every model (static, XGB-AFT, discrete-time
  hazard, RSF, GB-Cox, lifelines AFT) is fit on the **same** 25k subsample and
  scored on the **same** full test set - so differences are about the *model*, not
  the data volume. (In production the scalable models - static, XGB-AFT, hazard -
  would train on all ~109k rows and gain a little more.)
- **Two metrics, every model:** `cancel_AUC` is about general acccuracy and `C-index` is about timing.

In [2]:
# ---- build duration + event, then the temporal train/test split ----
df = load_clean_reservations().dropna(subset=["status"]).copy()
df["status"]  = df["status"].astype(int)
df["created"] = pd.to_datetime(df["created"], utc=True)
df["arrival"] = pd.to_datetime(df["arrival"], utc=True)

# cancel timestamp from RAW (cancellationTime is dropped from the clean frame as leakage)
raw = load_reservations()
m = raw[["arrival","created","cancellationTime","status"]].copy()
for col in ["arrival","created","cancellationTime"]:
    m[col] = pd.to_datetime(m[col], utc=True, errors="coerce")
m = m[m["status"]=="Canceled"].drop_duplicates(["arrival","created"])
df = df.merge(m[["arrival","created","cancellationTime"]], on=["arrival","created"], how="left")

# end of observation: cancel time (event) or arrival (censored). Missing cancel
# time on a cancelled row -> midpoint-of-lead-window fallback.
end = df["cancellationTime"].where(df["status"]==1, df["arrival"])
miss = (df["status"]==1) & end.isna()
end = end.copy()
end[miss] = df.loc[miss,"created"] + (df.loc[miss,"arrival"] - df.loc[miss,"created"]) / 2
df["cancel_ts"] = end.where(df["status"] == 1)   # cancel timestamp for events (NaT if censored)
df["duration"] = (end - df["created"]).dt.total_seconds() / 86400.0
# KEY RSF/GB-Cox MEMORY FIX: round duration to whole days. Fractional-day
# timestamps give ~tens-of-thousands of unique event times; scikit-survival
# forests store a step function at every unique training event time per node
# per tree (multi-GB). Integer days collapse that to ~90 unique times.
df["duration"] = df["duration"].round()
df["event"]    = df["status"].astype(bool)
df = df[df["duration"] > 0].copy()   # drop non-positive durations; min is now 1 day
print(f"rows {len(df):,} | events (cancellations) {df['event'].sum():,} "
      f"({df['event'].mean():.1%}) | median duration {df['duration'].median():.0f}d")

split_col = "temporal_split" if "temporal_split" in df.columns else None
if split_col:
    tr = df[df["temporal_split"]!="test"]; te = df[df["temporal_split"]=="test"]
else:  # fallback for an older parquet
    te = df[df["is_temporal_test"]==1]; tr = df[df["is_temporal_test"]==0]
Xtr, Xte = make_matrix(tr, te)
dur_tr, evt_tr = tr["duration"].to_numpy(), tr["event"].to_numpy()
dur_te, evt_te = te["duration"].to_numpy(), te["event"].to_numpy()
print(f"train {len(tr):,} | test {len(te):,}")

loading cached parquet: reservations_raw_no_pii.parquet


rows 144,983 | events (cancellations) 28,206 (19.5%) | median duration 12d
train 109,302 | test 35,681


In [3]:
import gc
from sklearn.metrics import roc_auc_score

def arrays(frame):
    lead = ((frame["arrival"] - frame["created"]).dt.total_seconds() / 86400).to_numpy()
    cdba = ((frame["arrival"] - frame["cancel_ts"]).dt.total_seconds() / 86400).to_numpy()  # NaN if censored
    return lead, cdba
lead_tr, cdba_tr = arrays(tr); lead_te, cdba_te = arrays(te)

def cindex(risk):
    risk = np.asarray(risk, dtype=float)
    if HAVE_SKSURV:   return concordance_index_censored(evt_te, dur_te, risk)[0]
    if HAVE_LIFELINES: return ll_concordance(dur_te, -risk, evt_te)
    return np.nan

results = {}
def score(name, risk_te):
    """risk_te: higher = more likely / sooner to cancel. Classic full-test metrics
    (cancel-AUC + C-index). These are SANITY numbers now - the decision table is the
    per-snapshot evaluation in the next cell."""
    risk_te = np.asarray(risk_te, dtype=float)
    auc = roc_auc_score(evt_te, risk_te)
    ci  = cindex(risk_te)
    results[name] = {"cancel_auc": auc, "c_index": ci}
    print(f"  {name:24s} cancel_AUC={auc:.4f}  C-index={ci:.4f}")

# ---- ONE common training sample for every model (fair head-to-head) ----
rng = np.random.default_rng(42)
BENCH_N = min(25000, len(tr))
bench = rng.choice(len(tr), size=BENCH_N, replace=False)
Xb = Xtr[bench].astype("float32", copy=False)
evt_b, dur_b = evt_tr[bench], dur_tr[bench]
lead_b, cdba_b = lead_tr[bench], cdba_tr[bench]

# ---- snapshot-eval bookkeeping: a common TEST subsample + a store of everything
# the per-snapshot cell needs (so heavy models can still be deleted after fitting).
EVAL_N   = min(20000, len(te))
eval_idx = np.sort(rng.choice(len(te), size=EVAL_N, replace=False))
X_eval   = Xte[eval_idx]
snap_store = {}        # name -> dict describing how to get P(cancel in (t1, t2] | alive)
SURV_GRID = np.arange(0.0, float(np.nanmax(lead_te)) + 2.0)  # day grid for survival curves

def surv_matrix(model, X):
    """(n, t) survival matrix + time grid from a scikit-survival model."""
    try:
        S = model.predict_survival_function(X, return_array=True)
        t = np.asarray(model.unique_times_, dtype=float)
    except Exception:
        fns = model.predict_survival_function(X)
        t = np.asarray(fns[0].x, dtype=float)
        S = np.vstack([fn(t) for fn in fns])
    return np.asarray(S, dtype="float32"), t

print(f"footprint -> common train {BENCH_N:,} (ALL models) | test {len(te):,} | "
      f"snapshot-eval subsample {EVAL_N:,} | unique event times {np.unique(dur_b).size}")

# 1) Static classifier - one P(cancel), frozen at booking time
clf = HistGradientBoostingClassifier(max_depth=8, learning_rate=0.05, max_iter=300, random_state=42)
clf.fit(Xb, evt_b.astype(int))
p_static = clf.predict_proba(Xte)[:, 1]
score("static_histgb", p_static)
snap_store["static_histgb"] = {"kind": "flat", "p": p_static}   # same number at every snapshot
del clf; gc.collect()

# 2) XGBoost AFT - parametric AFT; predict() returns exp(mu) (median survival time on
# the log-normal scale). With aft_loss_distribution=normal & scale sigma we get a full
# parametric survival curve: S(t) = 1 - Phi((ln t - mu) / sigma)  -> stash mu per booking.
AFT_SIGMA = 1.0
db = xgb.DMatrix(Xb); db.set_float_info("label_lower_bound", dur_b)
db.set_float_info("label_upper_bound", np.where(evt_b, dur_b, np.inf))
aft = xgb.train({"objective":"survival:aft","eval_metric":"aft-nloglik",
                 "aft_loss_distribution":"normal","aft_loss_distribution_scale":AFT_SIGMA,
                 "tree_method":"hist","max_depth":6,"learning_rate":0.05},
                db, num_boost_round=300)
aft_pred = aft.predict(xgb.DMatrix(Xte))            # exp(mu), in days
score("xgb_aft", -aft_pred)
snap_store["xgb_aft"] = {"kind": "aft_lognormal", "mu": np.log(np.clip(aft_pred, 1e-6, None)),
                         "sigma": AFT_SIGMA}
del aft, db; gc.collect()

# 3) Discrete-time hazard (notebook 08): snapshot expansion -> conditional hazards
SNAP = [90, 60, 30, 14, 7, 3, 1]
def hazard_rows(Xmat, lead, cdba, event, for_train):
    Xs, ys, bk, sn = [], [], [], []
    for i, s in enumerate(SNAP):
        s_next = SNAP[i + 1] if i + 1 < len(SNAP) else 0
        keep = (lead >= s) & (~event | (cdba < s)) if for_train else (lead >= s)
        idx = np.where(keep)[0]
        if len(idx) == 0: continue
        Xs.append(np.column_stack([Xmat[idx], np.full(len(idx), float(s), dtype="float32")]))
        bk.append(idx); sn.append(np.full(len(idx), s))
        if for_train:
            ys.append((event[idx] & (cdba[idx] < s) & (cdba[idx] >= s_next)).astype(int))
    X_exp = np.vstack(Xs).astype("float32")
    if for_train: return X_exp, np.concatenate(ys)
    return X_exp, np.concatenate(bk), np.concatenate(sn)

Xh, yh = hazard_rows(Xb, lead_b, cdba_b, evt_b, for_train=True)
hz = HistGradientBoostingClassifier(max_depth=8, learning_rate=0.05, max_iter=300,
                                    random_state=42).fit(Xh, yh)
Xh_te, bk_te, sn_te = hazard_rows(Xte, lead_te, cdba_te, evt_te, for_train=False)
h_te = hz.predict_proba(Xh_te)[:, 1]
surv = np.ones(len(te)); np.multiply.at(surv, bk_te, 1.0 - h_te)
score("discrete_time_hazard", 1.0 - surv)
# stash hazards as a (booking x snapshot) matrix for horizon-conditional chaining
H = np.full((len(te), len(SNAP)), np.nan, dtype="float32")
H[bk_te, [SNAP.index(s) for s in sn_te]] = h_te
snap_store["discrete_time_hazard"] = {"kind": "hazard_matrix", "H": H, "SNAP": SNAP}
del Xh, yh, hz, Xh_te, h_te, surv; gc.collect()

# 4) scikit-survival on the SAME common sample
if HAVE_SKSURV:
    y_b = np.array([(bool(e), float(d)) for e, d in zip(evt_b, dur_b)], dtype=[("event","?"),("time","<f8")])
    for name, model in [
        ("random_survival_forest", RandomSurvivalForest(n_estimators=60, min_samples_leaf=50,
            max_depth=8, max_features="sqrt", n_jobs=1, random_state=42)),
        ("gradient_boosted_cox", GradientBoostingSurvivalAnalysis(n_estimators=150,
            learning_rate=0.1, max_depth=3, subsample=0.5, random_state=42))]:
        try:
            model.fit(Xb, y_b); score(name, model.predict(Xte))
            S, t = surv_matrix(model, X_eval)               # survival curves, eval subsample only
            snap_store[name] = {"kind": "surv_grid", "S": S, "times": t, "eval_only": True}
        except Exception as e:
            results[name] = {"cancel_auc": np.nan, "c_index": np.nan}; print(f"  {name} failed: {e}")
        del model; gc.collect()
    del y_b; gc.collect()

# 5) lifelines parametric AFT on the SAME common sample
if HAVE_LIFELINES:
    fcols = [f"f{i}" for i in range(Xb.shape[1])]
    tr_df = pd.DataFrame(Xb, columns=fcols); tr_df["duration"] = dur_b; tr_df["event"] = evt_b.astype(int)
    te_df = pd.DataFrame(Xte, columns=fcols)
    for name, Fitter in [("weibull_aft", WeibullAFTFitter), ("lognormal_aft", LogNormalAFTFitter),
                         ("loglogistic_aft", LogLogisticAFTFitter)]:
        try:
            fitter = Fitter(penalizer=0.01).fit(tr_df, duration_col="duration", event_col="event")
            score(name, -fitter.predict_median(te_df).to_numpy())
            Sdf = fitter.predict_survival_function(te_df.iloc[eval_idx], times=SURV_GRID)
            snap_store[name] = {"kind": "surv_grid", "S": Sdf.to_numpy().T.astype("float32"),
                                "times": SURV_GRID, "eval_only": True}
            del fitter, Sdf
        except Exception as e:
            results[name] = {"cancel_auc": np.nan, "c_index": np.nan}; print(f"  {name} failed: {e}")
        gc.collect()
    del tr_df, te_df; gc.collect()

res = pd.DataFrame(results).T.sort_values("cancel_auc", ascending=False)
print("\nClassic full-test metrics (SANITY ONLY - decision table follows in the next cell):")
print(res.to_string(float_format=lambda v: f"{v:.4f}"))
from src import tables_dir
out = tables_dir() / "00_audit" / "survival_benchmark.csv"; out.parent.mkdir(parents=True, exist_ok=True)
res.to_csv(out); print(f"saved -> {out}")


footprint -> common train 25,000 (ALL models) | test 35,681 | snapshot-eval subsample 20,000 | unique event times 283
  static_histgb            cancel_AUC=0.9123  C-index=0.7953
  xgb_aft                  cancel_AUC=0.6967  C-index=0.8698
  discrete_time_hazard     cancel_AUC=0.8987  C-index=0.7635
  random_survival_forest   cancel_AUC=0.8579  C-index=0.8077
  gradient_boosted_cox     cancel_AUC=0.7360  C-index=0.8515
  weibull_aft              cancel_AUC=0.7362  C-index=0.8328
  lognormal_aft            cancel_AUC=0.7615  C-index=0.8385
  loglogistic_aft          cancel_AUC=0.7507  C-index=0.8383

Classic full-test metrics (SANITY ONLY - decision table follows in the next cell):
                        cancel_auc  c_index
static_histgb               0.9123   0.7953
discrete_time_hazard        0.8987   0.7635
random_survival_forest      0.8579   0.8077
lognormal_aft               0.7615   0.8385
loglogistic_aft             0.7507   0.8383
weibull_aft                 0.7362   0.8328
gr

## Per-snapshot evaluation - the decision table

For every snapshot **d days before arrival** we take the bookings that are still alive
at that moment (not yet cancelled, lead time >= d) and ask each model:
*"probability this booking cancels in the remaining d days?"*

- survival-curve models answer via the conditional survival ratio
  `P = 1 - S(t_arrival) / S(t_snapshot)`;
- the discrete hazard chains its per-snapshot hazards over the remaining grid points;
- the static classifier can only repeat its one frozen probability - its degradation
  at short horizons is exactly the gap a survival layer must close to earn its place.


In [4]:
# ---- per-snapshot decision metrics: AUC@d, Brier@d, expected-vs-realized@d ----
from sklearn.metrics import brier_score_loss

def surv_at(S, times, t):
    """Step-function read-off of survival curves: S row-wise, t per booking (days)."""
    j = np.clip(np.searchsorted(times, t, side="right") - 1, 0, len(times) - 1)
    out = S[np.arange(len(t)), j]
    return np.where(t < times[0], 1.0, out)         # before first event time: S = 1

# eval-subsample views of outcome arrays (all snap_store entries are aligned to these)
lead_e, cdba_e, evt_e = lead_te[eval_idx], cdba_te[eval_idx], evt_te[eval_idx]

from scipy.stats import norm
rows = []
for d in SNAP:                                       # 90, 60, ..., 1 days before arrival
    alive = (lead_e >= d) & ~(evt_e & (cdba_e >= d))  # exists already & not yet cancelled
    y     = (evt_e & (cdba_e < d))[alive].astype(int) # cancels in the remaining d days?
    if y.size == 0 or y.min() == y.max(): continue
    t2 = lead_e[alive]                               # elapsed days at arrival
    t1 = t2 - d                                      # elapsed days at the snapshot
    for name, st in snap_store.items():
        if st["kind"] == "flat":
            p = st["p"][eval_idx][alive]             # one frozen probability, every horizon
        elif st["kind"] == "aft_lognormal":
            mu = st["mu"][eval_idx][alive]; sg = st["sigma"]
            S1 = 1 - norm.cdf((np.log(np.clip(t1, 1e-6, None)) - mu) / sg)
            S1 = np.where(t1 <= 0, 1.0, S1)
            S2 = 1 - norm.cdf((np.log(np.clip(t2, 1e-6, None)) - mu) / sg)
            p  = 1 - S2 / np.clip(S1, 1e-9, None)
        elif st["kind"] == "hazard_matrix":
            Hm = st["H"][eval_idx][alive]            # (n_alive, n_snapshots)
            use = np.array(st["SNAP"]) <= d          # only the snapshots still ahead
            p  = 1 - np.exp(np.nansum(np.log1p(-np.clip(Hm[:, use], 0, 0.999999)), axis=1))
        elif st["kind"] == "surv_grid":
            S1 = surv_at(st["S"][alive], st["times"], t1)
            S2 = surv_at(st["S"][alive], st["times"], t2)
            p  = 1 - S2 / np.clip(S1, 1e-9, None)
        p = np.clip(np.nan_to_num(p, nan=float(y.mean())), 0.0, 1.0)
        rows.append({"snapshot_d": d, "model": name, "n": int(y.size),
                     "base_rate": float(y.mean()),
                     "auc": roc_auc_score(y, p),
                     "brier": brier_score_loss(y, p),
                     "expected": float(p.sum()), "realized": int(y.sum()),
                     "exp_over_real": float(p.sum() / max(y.sum(), 1))})

snap_res = pd.DataFrame(rows)
for metric, fmt in [("auc", "{:.4f}"), ("brier", "{:.4f}"), ("exp_over_real", "{:.2f}")]:
    print(f"\n=== {metric.upper()} per snapshot (columns = days before arrival) ===")
    print(snap_res.pivot(index="model", columns="snapshot_d", values=metric)
          .sort_index(axis=1, ascending=False)
          .to_string(float_format=lambda v: fmt.format(v)))

out2 = tables_dir() / "00_audit" / "survival_benchmark_snapshots.csv"
snap_res.to_csv(out2, index=False)
print(f"\nsaved -> {out2}")



=== AUC per snapshot (columns = days before arrival) ===
snapshot_d                 90     60     30     14     7      3      1 
model                                                                  
discrete_time_hazard   0.8719 0.8761 0.8922 0.9028 0.9026 0.8888 0.8615
gradient_boosted_cox   0.8440 0.8427 0.8562 0.8608 0.8578 0.8396 0.8052
loglogistic_aft        0.7934 0.7891 0.8069 0.8289 0.8393 0.8278 0.7958
lognormal_aft          0.7904 0.7838 0.8020 0.8235 0.8344 0.8237 0.7936
random_survival_forest 0.8189 0.8157 0.8301 0.8371 0.8282 0.8035 0.7759
static_histgb          0.8725 0.8761 0.8897 0.8979 0.8957 0.8735 0.8299
weibull_aft            0.7875 0.7856 0.8092 0.8292 0.8388 0.8242 0.7881
xgb_aft                0.8532 0.8594 0.8710 0.8843 0.8808 0.8605 0.8395

=== BRIER per snapshot (columns = days before arrival) ===
snapshot_d                 90     60     30     14     7      3      1 
model                                                                  
discrete_time_haza

## How to read it / verdict

- **The decision table is the per-snapshot one.** Look for the model with the best
  *short-horizon* (d <= 14) combination of AUC@d (separates the still-falling bookings),
  low Brier@d, and `exp_over_real` close to **1.00** (the portfolio forecast a revenue
  manager actually consumes). Long horizons (30-90d) matter less - decisions sharpen
  near arrival.
- **`exp_over_real` > 1** = the model over-forecasts remaining cancellations at that
  horizon (would push overbooking too hard); **< 1** = too timid. The static classifier
  is expected to drift > 1 at short horizons - it cannot condition on "still alive at d".
- **C-index / full-test cancel-AUC** are sanity numbers only; a model that wins C-index
  but loses Brier@7 loses, full stop.
- Adoption rule stays: a survival layer earns its place only if it beats the static
  baseline clearly on the short-horizon decision metrics - otherwise the discrete-time
  hazard (08) remains the designated dynamic layer.

*(Run order note: this notebook needs `Data/reservations_raw_no_pii.parquet` for
cancellation timestamps and re-uses the clean frame from 00.)*
